# Recommendation methods comparison: Archetype-only vs Ensemble vs Fused

On the same set of datasets we compare four ways to measure "recommendation → nearest-neighbor score":

1. **Archetype-only**: Assign the dataset's default config to an archetype, use that archetype's **general parameters** as the recommendation, then find the nearest-neighbor score in the DB.
2. **Ensemble**: Use the ensemble's **best_config** as the recommendation, then find the nearest-neighbor score in the DB.
3. **Fused-max**: For each dataset take the **maximum** of the Archetype score and the Ensemble score (take the better one).
4. **Fused-mean**: For each dataset take the **mean** of the two scores, then compute beat_default etc.

**Prerequisite:** Run `train_hp_meta_model` to produce `hp_meta_model/`, and use `hp_tuning_db_refined.csv`.

## 1. Configuration

In [1]:
import os
HP_DB_PATH = "hp_tuning_db_refined.csv"
MODEL_DIR = "hp_meta_model"
SEED = 42
# Whether to evaluate only on datasets in the testset dir (True=testset only, False=full or sample)
USE_TESTSET = True
TESTSET_DIR = "testset"
MAX_DATASETS = 0  # When USE_TESTSET=False; 0 = all

## 2. Dependencies and helper functions

In [2]:
import json
import pickle
import numpy as np
import pandas as pd
from train_hp_meta_model import HPEnsemblePredictor, HP_PARAM_NAMES

HP_BOUNDS = {
    "hp_num_leaves": (4, 256),
    "hp_max_depth": (3, 15),
    "hp_learning_rate": (0.005, 0.3),
    "hp_n_estimators": (50, 3000),
    "hp_min_child_samples": (5, 100),
    "hp_subsample": (0.5, 1.0),
    "hp_colsample_bytree": (0.3, 1.0),
    "hp_reg_alpha": (1e-8, 10.0),
    "hp_reg_lambda": (1e-8, 10.0),
    "hp_max_bin": (63, 511),
}

def config_to_hp_vector(config, keys):
    vec = []
    for k in keys:
        param = k.replace("hp_", "")
        val = config.get(param)
        if val is None:
            return None
        low, high = HP_BOUNDS[k]
        if param in ["learning_rate", "n_estimators", "reg_alpha", "reg_lambda"]:
            val = np.clip(val, low, high)
            log_low, log_high = np.log1p(low), np.log1p(high)
            n = (np.log1p(val) - log_low) / (log_high - log_low)
        else:
            n = (np.clip(val, low, high) - low) / (high - low) if high > low else 0.0
        vec.append(n)
    return np.array(vec, dtype=float)

def row_to_hp_vector(row, keys):
    vec = []
    for k in keys:
        val = row.get(k)
        if pd.isna(val):
            return None
        low, high = HP_BOUNDS[k]
        if k in ["hp_learning_rate", "hp_n_estimators", "hp_reg_alpha", "hp_reg_lambda"]:
            val = np.clip(float(val), low, high)
            log_low, log_high = np.log1p(low), np.log1p(high)
            n = (np.log1p(val) - log_low) / (log_high - log_low)
        else:
            val = np.clip(float(val), low, high)
            n = (val - low) / (high - low) if high > low else 0.0
        vec.append(n)
    return np.array(vec, dtype=float)

def hp_distance(v1, v2):
    return np.sqrt(np.sum((v1 - v2) ** 2))

def row_to_config(row, keys):
    """Convert a DB row (with hp_* cols) to config dict (no hp_ prefix)."""
    config = {}
    for k in keys:
        param = k.replace("hp_", "")
        val = row.get(k)
        if pd.isna(val):
            return None
        if param in ["num_leaves", "max_depth", "n_estimators", "min_child_samples", "max_bin"]:
            config[param] = int(round(float(val)))
        else:
            config[param] = float(val)
    return config

def assign_config_to_archetype(config, archetype_data):
    if archetype_data is None or "kmeans_model" not in archetype_data:
        return None
    kmeans = archetype_data["kmeans_model"]
    scaler = archetype_data["scaler"]
    hp_cols = archetype_data.get("hp_cols_used", [])
    if not hp_cols:
        return None
    vec = []
    for col in hp_cols:
        param = col.replace("hp_", "")
        val = config.get(param)
        if val is None:
            val = 0.0
        val = float(val)
        if param in ["learning_rate", "n_estimators", "reg_alpha", "reg_lambda"]:
            val = np.log1p(np.clip(val, 1e-10, None))
        vec.append(val)
    X = np.array([vec], dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    try:
        X_scaled = scaler.transform(X)
        lab = kmeans.predict(X_scaled)[0]
    except Exception:
        return "Unknown"
    cluster_id_to_name = {info["cluster_id"]: name for name, info in archetype_data.get("archetypes", {}).items()}
    return cluster_id_to_name.get(lab, "Unknown")

def get_archetype_params_as_config(archetype_data, archetype_name):
    """Get archetype's general parameters as config dict (aligned with HP cols)."""
    if not archetype_data or archetype_name is None:
        return None
    info = archetype_data.get("archetypes", {}).get(archetype_name)
    if not info:
        return None
    return info.get("params")

print("Helper functions defined.")

Helper functions defined.


## 3. Load DB and models, determine dataset list to evaluate

In [3]:
df = pd.read_csv(HP_DB_PATH)
print(f"HP DB: {len(df)} rows, {df['dataset_name'].nunique()} datasets")

ensemble = None
ensembles = {}
manifest_path = os.path.join(MODEL_DIR, "hp_manifest.json")
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    if manifest.get("per_task") and manifest.get("task_types"):
        for tt in manifest["task_types"]:
            subdir = os.path.join(MODEL_DIR, tt)
            if os.path.isdir(subdir) and os.path.isfile(os.path.join(subdir, "hp_scorer.pkl")):
                ensembles[tt] = HPEnsemblePredictor.load(MODEL_DIR, task_type=tt)
        print(f"Loaded by task type: {list(ensembles.keys())}")
if not ensembles:
    ensemble = HPEnsemblePredictor.load(MODEL_DIR)
    print("Single model loaded")

preparator = ensemble.preparator if ensemble is not None else list(ensembles.values())[0].preparator
hp_keys = [c for c in HP_PARAM_NAMES if c in HP_BOUNDS]
print(f"HP dim: {len(hp_keys)}")

archetypes_by_task = {}
if ensembles:
    for tt in ensembles:
        pkl_path = os.path.join(MODEL_DIR, tt, "hp_archetypes.pkl")
        if os.path.isfile(pkl_path):
            with open(pkl_path, "rb") as f:
                archetypes_by_task[tt] = pickle.load(f)
else:
    pkl_path = os.path.join(MODEL_DIR, "hp_archetypes.pkl")
    if os.path.isfile(pkl_path):
        with open(pkl_path, "rb") as f:
            archetypes_by_task["all"] = pickle.load(f)
print(f"Archetypes: {list(archetypes_by_task.keys())}")

# Dataset list to evaluate
if USE_TESTSET and os.path.isdir(TESTSET_DIR):
    stems = []
    for f in os.listdir(TESTSET_DIR):
        base, ext = os.path.splitext(f)
        if ext.lower() == ".csv" or not ext:
            stems.append(base)
    stems = list(dict.fromkeys(stems))
    db_names = df["dataset_name"].astype(str).unique()
    datasets = []
    for s in stems:
        for d in db_names:
            if d == s or d.startswith(s + "_") or d.startswith(s + "-"):
                datasets.append(d)
                break
    print(f"Testset matched {len(datasets)} datasets (stems: {stems})")
else:
    dataset_rows = df.drop_duplicates(subset="dataset_name", keep="first")
    datasets = dataset_rows["dataset_name"].tolist()
    if MAX_DATASETS > 0:
        rng = np.random.RandomState(SEED)
        idx = rng.permutation(len(datasets))[:MAX_DATASETS]
        datasets = [datasets[i] for i in idx]
    print(f"Will evaluate {len(datasets)} datasets")


HP DB: 135295 rows, 1749 datasets
Loaded by task type: ['classification', 'regression']
HP dim: 10
Archetypes: ['classification', 'regression']
Testset matched 8 datasets (stems: ['hiva_agnostic', 'maternal_health_risk', 'MIC', 'Amazon_employee_access', 'online_shoppers_intention', 'APSFailure', 'E-CommereShippingData', 'splice'])


## 4. Evaluate four methods per dataset

In [ ]:
def find_nearest_score_and_rank(rec_config, grp, hp_keys):
    """Find nearest neighbor of recommended config in grp; return (primary_score, rank)."""
    rec_vec = config_to_hp_vector(rec_config, hp_keys)
    if rec_vec is None:
        return None, None
    min_dist = np.inf
    chosen_primary = None
    chosen_idx = None
    for idx, (_, r) in enumerate(grp.iterrows()):
        rvec = row_to_hp_vector(r, hp_keys)
        if rvec is None:
            continue
        d = hp_distance(rec_vec, rvec)
        if d < min_dist:
            min_dist = d
            chosen_primary = float(r["primary_score"])
            chosen_idx = idx
    if chosen_primary is None:
        return None, None
    rank_series = grp["primary_score"].rank(ascending=False, method="min").astype(int)
    chosen_rank = int(rank_series.iloc[chosen_idx]) if chosen_idx is not None else None
    return chosen_primary, chosen_rank

results = []
for i, dname in enumerate(datasets):
    if (i + 1) % 20 == 0 or i == 0:
        print(f"  {i+1}/{len(datasets)} {str(dname)[:50]}...")
    grp = df[df["dataset_name"] == dname]
    if grp.empty:
        continue
    row0 = grp.iloc[0]
    task_type = str(row0.get("task_type", "classification")).strip().lower()
    if task_type not in ("classification", "regression"):
        task_type = "classification"

    if ensembles:
        ensemble_cur = ensembles.get(task_type)
        if ensemble_cur is None:
            continue
        preparator_cur = ensemble_cur.preparator
    else:
        ensemble_cur = ensemble
        preparator_cur = preparator

    arch_data = archetypes_by_task.get("all") if "all" in archetypes_by_task else archetypes_by_task.get(task_type)

    primary_scores = grp["primary_score"].values
    best_score = float(np.nanmax(primary_scores))
    default_row = grp[(grp.get("hp_is_default", pd.Series(0)) == 1)]
    if default_row.empty:
        default_row = grp[grp.get("hp_config_name", pd.Series("")) == "default"]
    default_score = float(default_row["primary_score"].iloc[0]) if len(default_row) > 0 else primary_scores[0]

    # --- Archetype-only: assign default to archetype → use that archetype's params as recommendation
    score_arch = None
    rank_arch = None
    if arch_data:
        def_row = default_row.iloc[0] if len(default_row) > 0 else grp.iloc[0]
        default_config = row_to_config(def_row.to_dict(), hp_keys)
        if default_config:
            arch_name = assign_config_to_archetype(default_config, arch_data)
            rec_arch = get_archetype_params_as_config(arch_data, arch_name)
            if rec_arch:
                score_arch, rank_arch = find_nearest_score_and_rank(rec_arch, grp, hp_keys)

    # --- Ensemble: use best_config as recommendation
    score_ens = None
    rank_ens = None
    meta_row = row0.to_dict()
    try:
        tt_enc = preparator_cur.task_type_encoder.transform([task_type])[0]
    except Exception:
        tt_enc = 0
    meta_row["task_type_encoded"] = tt_enc
    ref = preparator_cur.encode_refinement_dimensions(task_type, meta_row)
    for k, v in ref.items():
        meta_row[k] = v
    pred_df = pd.DataFrame([meta_row])
    if len([c for c in preparator_cur.feature_columns_predictor if c in pred_df.columns]) < len(preparator_cur.feature_columns_predictor):
        pred_df = pred_df.reindex(columns=preparator_cur.feature_columns_predictor)
        pred_df = pred_df.fillna(preparator_cur.fill_values)
    try:
        out = ensemble_cur.predict(pred_df, task_type=task_type, n_perturbations=20, seed=SEED)
        rec_ens = out["best_config"]
        score_ens, rank_ens = find_nearest_score_and_rank(rec_ens, grp, hp_keys)
    except Exception:
        pass

    if score_arch is None and score_ens is None:
        continue

    score_arch = score_arch if score_arch is not None else -np.inf
    score_ens = score_ens if score_ens is not None else -np.inf
    rank_arch = rank_arch if rank_arch is not None else np.nan
    rank_ens = rank_ens if rank_ens is not None else np.nan

    score_fused_max = max(score_arch, score_ens)
    rank_fused_max = rank_arch if score_arch >= score_ens else rank_ens
    score_fused_mean = (score_arch + score_ens) / 2

    results.append({
        "dataset": dname,
        "task_type": task_type,
        "default_score": default_score,
        "best_score": best_score,
        "score_arch": score_arch,
        "score_ens": score_ens,
        "score_fused_max": score_fused_max,
        "score_fused_mean": score_fused_mean,
        "rank_arch": rank_arch,
        "rank_ens": rank_ens,
        "rank_fused_max": rank_fused_max,
        "beat_default_arch": score_arch >= default_score,
        "beat_default_ens": score_ens >= default_score,
        "beat_default_fused_max": score_fused_max >= default_score,
        "beat_default_fused_mean": score_fused_mean >= default_score,
    })

res_df = pd.DataFrame(results)
print(f"\nDone, valid results: {len(res_df)} datasets.")

  1/8 hiva_agnostic...


AttributeError: 'HPDataPreparator' object has no attribute 'encode_refinement_dimensions'

## 5. Summary table and conclusions

In [ ]:
n = len(res_df)
if n == 0:
    print("No valid results.")
else:
    summary = []
    for method, col_score, col_beat, col_rank in [
        ("Archetype-only", "score_arch", "beat_default_arch", "rank_arch"),
        ("Ensemble", "score_ens", "beat_default_ens", "rank_ens"),
        ("Fused-max (take best)", "score_fused_max", "beat_default_fused_max", "rank_fused_max"),
        ("Fused-mean (average)", "score_fused_mean", "beat_default_fused_mean", None),
    ]:
        mean_score = res_df[col_score].mean()
        beat_pct = 100 * res_df[col_beat].mean()
        med_rank = res_df[col_rank].median() if col_rank and col_rank in res_df.columns else np.nan
        summary.append({
            "Method": method,
            "Mean rec score": round(mean_score, 6),
            "Beat default %": f"{beat_pct:.1f}%",
            "Median rank (1=best)": med_rank if pd.notna(med_rank) else "-",
        })
    sum_df = pd.DataFrame(summary)
    display(sum_df)

    print("\n--- Conclusions ---")
    best_mean = sum_df["Mean rec score"].max()
    best_row = sum_df.loc[sum_df["Mean rec score"] == best_mean].iloc[0]
    print(f"  Highest mean rec score: {best_row['Method']} ({best_mean:.6f})")
    best_beat = sum_df["Beat default %"].apply(lambda x: float(x.rstrip("%"))).max()
    best_beat_row = sum_df[sum_df["Beat default %"].str.rstrip("%").astype(float) == best_beat].iloc[0]
    print(f"  Highest beat default %: {best_beat_row['Method']} ({best_beat:.1f}%)")
    print("\n  Suggestion: If Archetype-only and Ensemble are close, prefer Archetype (interpretable); if Ensemble is clearly better, use Fused-max to combine both.")

,方法,平均推荐得分,Beat default %,中位排名(1=最优)
0,Archetype-only,0.826997,100.0%,61.0
1,Ensemble,0.834263,87.5%,31.0
2,Fused-max (取优),0.834376,100.0%,31.0
3,Fused-mean (平均),0.830630,87.5%,-



--- 结论 ---
  平均推荐得分最高: Fused-max (取优) (0.834376)
  Beat default 占比最高: Archetype-only (100.0%)

  建议: 若 Archetype-only 与 Ensemble 接近，可优先 Archetype（可解释）；若 Ensemble 明显更好，可用 Fused-max 兼顾两者。
